In [5]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [4]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)


In [7]:
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

In [9]:
import sys
import os

# Go up one folder to the project root
sys.path.append(os.path.abspath(".."))

In [10]:
from part_1_RAG.rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI


ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required by the SDK but ignored by Ollama
)

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=ollama_client,
)

In [11]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Based on the provided context, **yes**, you can still sign up (join) even if the course has started or you just discovered it.\n\nHowever, please note the following conditions regarding your sign-up and certification:\n\n*   **Certificate Requirement:** If you want to receive a certificate, you must submit your project while submissions are still being accepted.\n*   **Optional Registration:** You can start learning and submit homework without formally registering. Registration is primarily used to gauge interest before the start date.\n*   **Submission Window:** If you cannot participate in the first submission window (attempt #1), there is still time to catch up during the second submission window (attempt #2).'

In [13]:
vs_index.close()